# 03 — Dictionaries, Associative Mapping, and Hash Tables

## Theoretical Foundations of Dictionaries

In computer science, an **associative array** or **dictionary** represents an abstract data type composed of a collection of unique keys and their mapped values:

$$f: K \to V, \quad \text{where } k_i \neq k_j \text{ for all } i \neq j$$

In Python, the `dict` type is implemented as an optimized hash table.

### CPython Compact Hash Table Architecture (Python 3.6+ / PEP 468)

Prior to Python 3.6, dictionaries allocated a large sparse array of 24-byte entries `(hash, key, value)`. Unused buckets consumed significant memory.

Since Python 3.6, CPython employs a **compact dictionary** design:
1. **Dense `entries` array:** Key-value pairs are stored sequentially in an array of `PyDictKeyEntry` structs in the exact order they are inserted:
   `entries = [(hash_0, key_0, val_0), (hash_1, key_1, val_1), ...]`
2. **Sparse `indices` array:** A small hash table array containing indices pointing into the dense `entries` array.

This architecture delivers two major benefits:
- **Memory reduction:** Reduces dictionary memory footprint by 20% to 25%.
- **Deterministic ordering:** Dictionaries naturally preserve insertion order as a language guarantee.


In [ ]:
# 1. Dictionary Construction Syntax
# Direct literal syntax
model_config = {
    "architecture": "ResNet-50",
    "batch_size": 32,
    "learning_rate": 0.001,
}

# Constructor syntax with keyword arguments
optimizer_config = {"name": "AdamW", "weight_decay": 1e-4, "eps": 1e-8}

# Constructing from an iterable of key-value pairs
loss_weights = {"cross_entropy": 1.0, "dice_loss": 0.5}

# Constructing from zipped sequences
hyper_keys = ["lr", "momentum", "decay"]
hyper_values = [0.01, 0.9, 0.0005]
hyperparams = dict(zip(hyper_keys, hyper_values, strict=True))

# Constructing with default values via fromkeys()
default_flags = dict.fromkeys(["train", "val", "test"], False)

print(f"Model config:     {model_config}")
print(f"Optimizer config: {optimizer_config}")
print(f"Hyperparams:      {hyperparams}")
print(f"Default flags:    {default_flags}")

assert hyperparams["lr"] == 0.01
assert default_flags["train"] is False

## Safe Querying, Dynamic Updates, and Key Eviction

Accessing non-existent keys via direct indexing (`d[k]`) raises a `KeyError`. Python provides defensive idioms:
- `d.get(k, default)`: Returns mapped value if `k in d`; returns `default` (or `None`) without raising exceptions.
- `d.setdefault(k, default)`: Returns value if `k in d`; otherwise sets `d[k] = default` and returns `default`.
- `d.pop(k, default)`: Removes and returns `d[k]`; returns `default` if absent.
- `d.popitem()`: Removes and returns the last inserted `(key, value)` pair in LIFO order.
- `del d[k]`: Deletes key `k`; raises `KeyError` if absent.


In [ ]:
# 2. Safe Querying and Dynamic Mutation
settings = {"learning_rate": 0.05, "epochs": 100}

# 1. get() defensive lookup
warmup = settings.get("warmup_steps", 500)
assert warmup == 500
assert "warmup_steps" not in settings

# 2. setdefault() lookup and auto-initialization
decay = settings.setdefault("weight_decay", 1e-4)
assert decay == 1e-4
assert "weight_decay" in settings
assert settings["weight_decay"] == 1e-4

# 3. pop() with eviction
removed_epochs = settings.pop("epochs")
assert removed_epochs == 100
assert "epochs" not in settings

# 4. popitem() removes the most recently added key
last_key, last_val = settings.popitem()
print(f"Removed via popitem(): {last_key} -> {last_val}")
assert last_key == "weight_decay"

print(f"Remaining settings: {settings}")

## Dynamic Dictionary Views

Methods `keys()`, `values()`, and `items()` return **dynamic view objects**:
- They provide a read-only, real-time window into the dictionary's contents without allocating new auxiliary lists.
- Mutations in the dictionary are immediately reflected in existing view references.
- `keys()` and `items()` views support set-like algebra (`&`, `|`, `-`).


In [ ]:
# 3. Dynamic Dictionary Views
registry = {"model_a": 0.88, "model_b": 0.91}

keys_view = registry.keys()
items_view = registry.items()

print(f"Initial keys: {list(keys_view)}")

# Mutating dictionary in-place
registry["model_c"] = 0.94

# View immediately reflects mutation
print(f"Dynamically updated keys: {list(keys_view)}")
assert "model_c" in keys_view

# Set operations on dictionary keys
allowed_models = {"model_b", "model_c", "model_d"}
common_models = keys_view & allowed_models
assert common_models == {"model_b", "model_c"}
print(f"Common models via keys & set: {common_models}")

## Modern Dictionary Merging (PEP 584) and Comprehensions

Python 3.9 introduced dedicated dictionary merge (`|`) and update (`|=`) operators:
- `d1 | d2`: Evaluates a new dictionary containing merged keys. When keys collide, right-hand operand values overwrite left-hand values.
- `d1 |= d2`: Mutates `d1` in-place by updating with pairs from `d2`.


In [ ]:
# 4. Dictionary Merging and Comprehensions
baseline_params = {"lr": 0.01, "batch_size": 32, "dropout": 0.2}
tuned_params = {"lr": 0.005, "dropout": 0.5, "optimizer": "Adam"}

# Merge operator |
active_params = baseline_params | tuned_params
print(f"Merged parameters: {active_params}")

assert active_params["lr"] == 0.005  # Right operand overwrote left
assert active_params["batch_size"] == 32  # Preserved from left
assert active_params["optimizer"] == "Adam"  # Introduced from right

# Dictionary comprehension: filtering and transforming
high_precision_params = {k: v for k, v in active_params.items() if isinstance(v, float) and v < 0.1}
print(f"High precision params: {high_precision_params}")
assert high_precision_params == {"lr": 0.005}

## Persistence: Loading and Updating JSON Configurations

Dictionaries serve as the standard in-memory representation for JSON records. Here, we ingest `../data/data.json`, inspect training parameters, apply defensive validations, and serialize an updated configuration.


In [ ]:
# 5. JSON Deserialization and Persistence
import json
from pathlib import Path

json_file = Path("../data/data.json")
with json_file.open("r", encoding="utf-8") as f:
    training_spec = json.load(f)

print(f"Loaded JSON specification: {training_spec}")
assert "learning_rate" in training_spec
assert training_spec["optimizer"] == "Adam"

# Defensive update
experiment_spec = training_spec.copy()
experiment_spec["learning_rate"] = 0.01
experiment_spec["batch_size"] = 128
experiment_spec["scheduler"] = "CosineAnnealing"

print(f"Experiment configuration: {experiment_spec}")
serialized_json = json.dumps(experiment_spec, indent=2)
print(f"Serialized output preview:\n{serialized_json}")

## Case Study: Term Frequency Analysis (Bag-of-Words)

We implement term frequency counting across the text in `../data/data.txt` using pure Python dictionary idioms (`get()` and comprehensions), analyzing word occurrences.


In [ ]:
# 6. Term Frequency Counter Case Study
from pathlib import Path

corpus = Path("../data/data.txt").read_text(encoding="utf-8")
tokens = [w.strip(".,;:()-/\"'").lower() for w in corpus.split() if w.strip(".,;:()-/\"'")]

# Frequency accumulation via dict.get()
term_frequencies: dict[str, int] = {}
for token in tokens:
    term_frequencies[token] = term_frequencies.get(token, 0) + 1

# Rank by frequency descending
ranked_terms = sorted(term_frequencies.items(), key=lambda pair: pair[1], reverse=True)

print("Top 5 most frequent terms:")
for rank, (term, count) in enumerate(ranked_terms[:5], start=1):
    print(f"  {rank}. '{term}': {count} occurrences")

assert term_frequencies["python"] >= 2
assert term_frequencies["programming"] >= 2